# Debugger and Profiler (SageMaker Python SDK v3)

**What changed vs. the original course notebook**

| v2 (course) | v3 (this notebook) |
|---|---|
| `sagemaker.pytorch.PyTorch(...)` estimator | `sagemaker.train.ModelTrainer(...)` |
| `entry_point`, `instance_type`, `framework_version` kwargs | `SourceCode(...)`, `Compute(...)`, explicit `training_image` |
| `estimator.fit()` | `trainer.train()` |
| `from sagemaker.debugger import ...` | `from sagemaker.core.debugger import ...` |
| `estimator.latest_training_job`, `latest_job_debugger_artifacts_path()`, `rule_job_summary()` | plain `boto3` `describe_training_job` + S3 paths we define ourselves |

> **Heads-up:** AWS has closed SageMaker Debugger to new customers (existing customers can keep using it).
> If job creation fails with a Debugger/rules error, your account is probably affected. AWS recommends
> MLflow / TensorBoard / CloudWatch as replacements.

In [1]:
# Upgrade to SDK v3 + smdebug (client library used to read the tensors back). Restart the kernel afterwards.
!pip install -U "sagemaker>=3.0" smdebug

  Using cached smdebug-1.0.34-py2.py3-none-any.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of smdebug to determine which version is compatible with other requirements. This could take a while.


In [2]:
hyperparameters = {
    "batch_size": 2048,
    "gpu": True,
    "epoch": 2,
    "model": "resnet50",
}

## Session, role, S3 locations and source directory

In [3]:
import os

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()
role = get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()

# Everything (model, Debugger tensors, profiler data, rule output) goes under this prefix
output_s3 = f"s3://{bucket}/cifar-profiling"

# ModelTrainer uploads a whole *directory* to the training job: ./scripts (where your script lives).
source_dir = "scripts"
assert os.path.exists(f"{source_dir}/pytorch_cifar_profiling.py"), "expected scripts/pytorch_cifar_profiling.py"

# Create scripts/requirements.txt so the container pip-installs smdebug before running the script
# (the script imports smdebug). It is created here, so you don't need to make it yourself.
with open(f"{source_dir}/requirements.txt", "w") as f:
    f.write("smdebug\n")

# v3 has no PyTorch estimator, so we point at the AWS Deep Learning Container ourselves.
# PyTorch 1.13 is used on purpose: smdebug's hook support targets the older 1.x line.
# (763104351884 is the DLC registry account for standard commercial regions.)
training_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:1.13.1-gpu-py39"

print(role)
print(output_s3)
print(training_image)

arn:aws:iam::950885994135:role/service-role/AmazonSageMaker-ExecutionRole-20261013T173018
s3://sagemaker-us-east-1-950885994135/cifar-profiling
763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:1.13.1-gpu-py39


In [4]:
from sagemaker.core.debugger import Rule, ProfilerRule, rule_configs

# Debugger rules inspect the saved tensors; ProfilerRule inspects system metrics.
# ProfilerReport is what generates the profiler-report.html used at the end of this notebook.
rules = [
    ProfilerRule.sagemaker(rule_configs.ProfilerReport()),
    Rule.sagemaker(rule_configs.loss_not_decreasing()),
    Rule.sagemaker(rule_configs.vanishing_gradient()),
    Rule.sagemaker(rule_configs.overfit()),
    Rule.sagemaker(rule_configs.overtraining()),
]

In [5]:
from sagemaker.core.debugger import DebuggerHookConfig, ProfilerConfig, CollectionConfig

# System profiling (CPU/GPU/memory/IO) sampled every 500 ms.
# Framework profiling (FrameworkProfile) is deprecated for PyTorch >= 2.0, so it is left out.
profiler_config = ProfilerConfig(
    s3_output_path=output_s3,
    system_monitor_interval_millis=500,
)

# Which tensors the hook saves and how often. With batch_size=2048 CIFAR-10 has only
# ~25 train steps and ~5 eval steps per epoch, so the intervals must be small or you'd
# end up with almost no datapoints.
debugger_config = DebuggerHookConfig(
    s3_output_path=output_s3,
    hook_parameters={"train.save_interval": "5", "eval.save_interval": "1"},
    collection_configs=[
        CollectionConfig(name="losses"),      # needed by loss_not_decreasing / overfit / overtraining
        CollectionConfig(name="gradients"),   # needed by vanishing_gradient
    ],
)

## Create the trainer

`ModelTrainer` replaces the `PyTorch` estimator. The AWS documentation shows Debugger/Profiler settings being passed
to `ModelTrainer` as `profiler_config=`, `debugger_hook_config=` and `rules=`. The guard below checks that **your
installed SDK version** actually accepts them before launching a billable job.

In [6]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, OutputDataConfig

# Hyperparameters are passed to the script explicitly as CLI args (argparse in the script)
command = "python pytorch_cifar_profiling.py " + " ".join(f"--{k} {v}" for k, v in hyperparameters.items())

debugger_kwargs = dict(
    profiler_config=profiler_config,
    debugger_hook_config=debugger_config,
    rules=rules,
)

unsupported = [k for k in debugger_kwargs if k not in ModelTrainer.model_fields]
if unsupported:
    print("ModelTrainer fields in this SDK version:", sorted(ModelTrainer.model_fields))
    raise RuntimeError(
        f"This SDK version's ModelTrainer has no {unsupported} parameter(s), so the Debugger/Profiler "
        "configuration cannot be attached through it. See the printed field list above."
    )

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    sagemaker_session=sagemaker_session,
    base_job_name="cifar-profiling",
    source_code=SourceCode(
        source_dir=source_dir,
        requirements="requirements.txt",
        command=command,
    ),
    compute=Compute(instance_type="ml.g4dn.xlarge", instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=output_s3),
    hyperparameters=hyperparameters,
    **debugger_kwargs,
)

ModelTrainer fields in this SDK version: ['algorithm_name', 'base_job_name', 'checkpoint_config', 'compute', 'config_mgr', 'distributed', 'environment', 'hyperparameters', 'input_data_config', 'input_s3_key_prefix', 'local_container_root', 'model_package_config', 'networking', 'output_data_config', 'role', 'sagemaker_session', 'source_code', 'stopping_condition', 'tags', 'training_image', 'training_image_config', 'training_input_mode', 'training_mode']


RuntimeError: This SDK version's ModelTrainer has no ['profiler_config', 'debugger_hook_config', 'rules'] parameter(s), so the Debugger/Profiler configuration cannot be attached through it. See the printed field list above.

In [ ]:
trainer.train(wait=True)

In [ ]:
training_job_name = trainer._latest_training_job.training_job_name
print(f"Training jobname: {training_job_name}")
print(f"Region: {region}")

# v3 replacement for the Estimator helpers: ask the SageMaker API directly
sm_client = boto3.client("sagemaker", region_name=region)
job_description = sm_client.describe_training_job(TrainingJobName=training_job_name)

In [ ]:
from smdebug.trials import create_trial
from smdebug.core.modes import ModeKeys

# Replaces estimator.latest_job_debugger_artifacts_path()
debugger_artifacts_path = f"{job_description['DebugHookConfig']['S3OutputPath'].rstrip('/')}/{training_job_name}/debug-output"
print(debugger_artifacts_path)

trial = create_trial(debugger_artifacts_path)

In [ ]:
# Names of all the tensors that were tracked
all_tensors = trial.tensor_names()
print(f"{len(all_tensors)} tensors tracked:")
for name in all_tensors:
    print("  ", name)

# Number of datapoints for one tensor, in train and eval mode.
# Use the loss tensor (registered via hook.register_loss) rather than an arbitrary weight tensor.
loss_tensor_name = trial.tensor_names(collection="losses")[0]
train_steps = trial.tensor(loss_tensor_name).steps(mode=ModeKeys.TRAIN)
eval_steps = trial.tensor(loss_tensor_name).steps(mode=ModeKeys.EVAL)

print(f"\nTensor: {loss_tensor_name}")
print(f"  datapoints in TRAIN mode: {len(train_steps)}")
print(f"  datapoints in EVAL mode : {len(eval_steps)}")

In [ ]:
from smdebug.profiler.analysis.notebook_utils.training_job import TrainingJob

tj = TrainingJob(training_job_name, region)
tj.wait_for_sys_profiling_data_to_be_available()

In [ ]:
from smdebug.profiler.analysis.notebook_utils.timeline_charts import TimelineCharts

system_metrics_reader = tj.get_systems_metrics_reader()
system_metrics_reader.refresh_event_file_list()

view_timeline_charts = TimelineCharts(
    system_metrics_reader,
    framework_metrics_reader=None,
    select_dimensions=["CPU", "GPU"],
    select_events=["total"],
)

In [ ]:
# Rule output lives under <OutputDataConfig S3 path>/<job name>/rule-output
rule_output_path = f"{output_s3}/{training_job_name}/rule-output"
print(f"You will find the profiler report in {rule_output_path}")

In [ ]:
! aws s3 ls {rule_output_path} --recursive

In [ ]:
! aws s3 cp {rule_output_path} ./ --recursive

In [ ]:
# Re-describe the job: rule evaluation statuses are only complete once rule jobs have finished
job_description = sm_client.describe_training_job(TrainingJobName=training_job_name)

# get the autogenerated folder name of profiler report (replaces estimator.latest_training_job.rule_job_summary())
profiler_report_name = [
    rule["RuleConfigurationName"]
    for rule in job_description["ProfilerRuleEvaluationStatuses"]
    if "Profiler" in rule["RuleConfigurationName"]
][0]
print(profiler_report_name)

In [ ]:
import IPython

IPython.display.HTML(filename=profiler_report_name + "/profiler-output/profiler-report.html")